In [1]:
from arcgis.gis import GIS
from arcgis.graph import KnowledgeGraph, Entity, Relationship, EntityType, RelationshipType, GraphProperty
import arcpy
import uuid

In [2]:
streetsFC   = r'C:\EsriTraining\PSAW\Instructor\PSAW_12.0_Aug25_Demos\DemoData\PSAW\PSAW\SanFran\SanFrancisco.gdb\Transportation\Streets'
junctionsFC = r'C:\EsriTraining\PSAW\Instructor\PSAW_12.0_Aug25_Demos\DemoData\PSAW\PSAW\SanFran\SanFrancisco.gdb\Transportation\Streets_ND_Junctions'
ndsPath     = r'C:\EsriTraining\PSAW\Instructor\PSAW_12.0_Aug25_Demos\DemoData\PSAW\PSAW\SanFran\SanFrancisco.gdb\Transportation\Streets_ND'

#### Create knowledge graph service

In [3]:
# ════════════════════════════════════════════════════════════
# PHASE 1 — Create a new, empty Knowledge Graph
# ════════════════════════════════════════════════════════════

gis = GIS("Pro")

kgService = gis.content.create_service(
    name="SFTransportation",
    service_type="KnowledgeGraph"
)

print(f"✅ Service created: {kgService.title}")
print(f"   URL: {kgService.url}")

kg = KnowledgeGraph(kgService.url, gis=gis)

✅ Service created: SFTransportation
   URL: https://ebase.ad.local/server/rest/services/Hosted/SFTransportation/KnowledgeGraphServer


In [4]:
# ════════════════════════════════════════════════════════════
# PHASE 2 — Build the data model
# ════════════════════════════════════════════════════════════

# ── Junction entity type ───────────────────────────────────
junctionType = EntityType(
    name="Junction",
    alias="Junction",
    properties=[
        GraphProperty(
            name="junctionEid",
            alias="Junction EID",
            field_type="esriFieldTypeInteger",
            not_nullable=True,
            not_editable=False,
            visible=True
        )
    ]
)

# ── Street entity type ────────────────────────────────────
streetType = EntityType(
    name="Street",
    alias="Street",
    properties=[
        GraphProperty(
            name="edgeEid",
            alias="Edge EID",
            field_type="esriFieldTypeInteger",
            not_nullable=True,
            not_editable=False,
            visible=True
        ),
        GraphProperty(
            name="sourceOid",
            alias="Source OID",
            field_type="esriFieldTypeInteger",
            not_nullable=False,
            not_editable=False,
            visible=True
        )
    ]
)

# ── Relationship types ─────────────────────────────────────
startsAtType = RelationshipType(
    name="StartsAt",
    alias="Starts At",
    origin_entity_types=["Street"],
    destination_entity_types=["Junction"],
    properties=[]
)

endsAtType = RelationshipType(
    name="EndsAt",
    alias="Ends At",
    origin_entity_types=["Street"],
    destination_entity_types=["Junction"],
    properties=[]
)

connectsToType = RelationshipType(
    name="ConnectsTo",
    alias="Connects To",
    origin_entity_types=["Street"],
    destination_entity_types=["Street"],
    properties=[]
)

# ── Apply the data model ──────────────────────────────────
kg.named_object_type_adds(
    entity_types=[junctionType, streetType],
    relationship_types=[startsAtType, endsAtType, connectsToType]
)

print("✅ Data model created: Junction, Street, StartsAt, EndsAt, ConnectsTo")

# ── Verify data model before populating ────────────────────
dm = kg.datamodel
requiredTypes = ["Junction", "Street", "StartsAt", "EndsAt", "ConnectsTo"]
existingTypes = list(dm["entity_types"].keys()) + list(dm["relationship_types"].keys())
missing = [t for t in requiredTypes if t not in existingTypes]

if missing:
    raise RuntimeError(f"❌ Data model incomplete — missing: {missing}. "
                       f"Re-run Phase 2 before proceeding.")
else:
    print("✅ Data model verified. Proceeding to populate.")

✅ Data model created: Junction, Street, StartsAt, EndsAt, ConnectsTo
✅ Data model verified. Proceeding to populate.


C:\Users\tim10393\AppData\Local\Temp\1\ipykernel_15124\1481390141.py:71: DeprecationWarning: In the future, the as_dict parameter will be removed, and the behavior will be as though as_dict is False. Setting as_dict to False is recommended.
  kg.named_object_type_adds(


In [5]:
# ════════════════════════════════════════════════════════════
# PHASE 3 — Iterate the network dataset and populate the KG
# ════════════════════════════════════════════════════════════

nds = arcpy.nax.NetworkDataset(ndsPath)

junctionUuids     = {}
streetUuids       = {}
junctionAdds      = []
streetAdds        = []
relationshipAdds  = []
edgeRecords       = []
streetsStartingAt = {}

# ── 3a. Iterate edges — collect entities + relationships ───

for eid, sourceOid, fromJ, toJ in nds.edges(
    ["EID", "SOURCEOID", "FROMJUNCTION", "TOJUNCTION"]
):
    edgeRecords.append((eid, sourceOid, fromJ, toJ))

    # ── Street entity ─────────────────────────────────────
    streetGuid = '{'+str(uuid.uuid4())+'}'
    streetUuids[eid] = streetGuid

    streetAdds.append(
        Entity(
            type_name="Street",
            global_id=streetGuid,
            properties={
                "edgeEid": int(eid),
                "sourceOid": int(sourceOid)
            }
        )
    )

    # ── Junction entities (deduplicate on the fly) ────────
    if fromJ not in junctionUuids:
        fromGuid = '{'+str(uuid.uuid4())+'}'
        junctionUuids[fromJ] = fromGuid
        junctionAdds.append(
            Entity(
                type_name="Junction",
                global_id=fromGuid,
                properties={"junctionEid": int(fromJ)}
            )
        )

    if toJ not in junctionUuids:
        toGuid = '{'+str(uuid.uuid4())+'}'
        junctionUuids[toJ] = toGuid
        junctionAdds.append(
            Entity(
                type_name="Junction",
                global_id=toGuid,
                properties={"junctionEid": int(toJ)}
            )
        )

    # ── StartsAt ──────────────────────────────────────────
    relationshipAdds.append(
        Relationship(
            type_name="StartsAt",
            global_id='{'+str(uuid.uuid4())+'}',
            origin_entity_id=streetGuid,
            origin_entity_type="Street",
            destination_entity_id=junctionUuids[fromJ],
            destination_entity_type="Junction",
            properties={}
        )
    )

    # ── EndsAt ────────────────────────────────────────────
    relationshipAdds.append(
        Relationship(
            type_name="EndsAt",
            global_id='{'+str(uuid.uuid4())+'}',
            origin_entity_id=streetGuid,
            origin_entity_type="Street",
            destination_entity_id=junctionUuids[toJ],
            destination_entity_type="Junction",
            properties={}
        )
    )

    # ── Track which streets start at each junction ────────
    streetsStartingAt.setdefault(fromJ, []).append(streetGuid)

# ── 3b. Build ConnectsTo relationships (second pass) ──────
connectsToAdds = []

for eid, sourceOid, fromJ, toJ in edgeRecords:
    originGuid = streetUuids[eid]

    for destGuid in streetsStartingAt.get(toJ, []):
        if destGuid == originGuid:
            continue
        connectsToAdds.append(
            Relationship(
                type_name="ConnectsTo",
                global_id='{'+str(uuid.uuid4())+'}',
                origin_entity_id=originGuid,
                origin_entity_type="Street",
                destination_entity_id=destGuid,
                destination_entity_type="Street",
                properties={}
            )
        )

print(f"Unique junctions   : {len(junctionAdds)}")
print(f"Street edges       : {len(streetAdds)}")
print(f"StartsAt / EndsAt  : {len(relationshipAdds)}")
print(f"ConnectsTo         : {len(connectsToAdds)}")

# ── 3c. Batch apply edits with error surfacing ─────────────
batchSize = 2000

def applyInBatches(records, label):
    total = len(records)
    if total == 0:
        print(f"  {label}: no records to add, skipping.")
        return
    for i in range(0, total, batchSize):
        batch = records[i : i + batchSize]
        if not batch:
            continue
        result = kg.apply_edits(adds=batch)

        # ── Surface any errors ────────────────────────────
        hasErrors = False
        if isinstance(result, dict):
            editResults = result.get("editResults", [])
            errors = [r for r in editResults if not r.get("success", True)]
            if errors:
                hasErrors = True
                print(f"  ⚠️ {label}: batch {i // batchSize + 1} — "
                      f"{len(errors)} failures out of {len(batch)}")
                print(f"     First error: {errors[0]}")

        if not hasErrors:
            print(f"  {label}: batch {i // batchSize + 1} "
                  f"({len(batch)} records, "
                  f"{min(i + batchSize, total)}/{total})")

# ── 3d. Write to KG ───────────────────────────────────────
applyInBatches(junctionAdds, "Junctions")
applyInBatches(streetAdds, "Streets")
applyInBatches(relationshipAdds, "StartsAt/EndsAt")
applyInBatches(connectsToAdds, "ConnectsTo")

print("✅ Knowledge graph populated.")

Unique junctions   : 37695
Street edges       : 108226
StartsAt / EndsAt  : 216452
ConnectsTo         : 344140
  Junctions: batch 1 (2000 records, 2000/37695)


C:\Users\tim10393\AppData\Local\Temp\1\ipykernel_15124\2683020983.py:127: DeprecationWarning: In the future, the as_dict parameter will be removed, and the behavior will be as though as_dict is False. Setting as_dict to False is recommended.
  result = kg.apply_edits(adds=batch)


  Junctions: batch 2 (2000 records, 4000/37695)
  Junctions: batch 3 (2000 records, 6000/37695)
  Junctions: batch 4 (2000 records, 8000/37695)
  Junctions: batch 5 (2000 records, 10000/37695)
  Junctions: batch 6 (2000 records, 12000/37695)
  Junctions: batch 7 (2000 records, 14000/37695)
  Junctions: batch 8 (2000 records, 16000/37695)
  Junctions: batch 9 (2000 records, 18000/37695)
  Junctions: batch 10 (2000 records, 20000/37695)
  Junctions: batch 11 (2000 records, 22000/37695)
  Junctions: batch 12 (2000 records, 24000/37695)
  Junctions: batch 13 (2000 records, 26000/37695)
  Junctions: batch 14 (2000 records, 28000/37695)
  Junctions: batch 15 (2000 records, 30000/37695)
  Junctions: batch 16 (2000 records, 32000/37695)
  Junctions: batch 17 (2000 records, 34000/37695)
  Junctions: batch 18 (2000 records, 36000/37695)
  Junctions: batch 19 (1695 records, 37695/37695)
  Streets: batch 1 (2000 records, 2000/108226)
  Streets: batch 2 (2000 records, 4000/108226)
  Streets: batch 

for eid, sourceOID, fromJ, toJ in nds.edges(["EID", "SOURCEOID", "FROMJUNCTION", "TOJUNCTION"]):
    print(F"ID of edge: {eid}, Feature OID: {sourceOID}, Origin edge: {fromJ}, Destination edge: {toJ}")

In [10]:
%reset -f